# Validação e Serialização com Pydantic

Este notebook explora funcionalidades avançadas do **Pydantic**, incluindo validadores de pré e pós-modelo, além de serializadores personalizados para controlar a saída dos dados (especialmente para JSON).

## 1. Importações

Importamos as ferramentas necessárias, incluindo `Self` para tipagem e novos decoradores de serialização do Pydantic.

In [1]:
import enum
import hashlib
import re
from typing import Any, Self
from pydantic import (
    BaseModel,
    EmailStr,
    Field,
    field_serializer,
    field_validator,
    model_serializer,
    model_validator,
    SecretStr,
)

## 2. Configurações de Validação

Definição das expressões regulares para garantir a integridade de nomes e senhas.

In [2]:
VALID_PASSWORD_REGEX = re.compile(r"^(?=.*[a-z])(?=.*[A-Z])(?=.*\d).{8,}$")
VALID_NAME_REGEX = re.compile(r"^[a-zA-Z]{2,}$")

## 3. Enumeração de Papéis (Roles)

Utilizamos `IntFlag` para gerir permissões. Note a inclusão do papel base `User` com valor 0.

In [3]:
class Role(enum.IntFlag):
    User = 0
    Author = 1
    Editor = 2
    Admin = 4
    SuperAdmin = 8

## 4. Modelo User com Lógica Avançada

Este modelo demonstra:
- **`exclude=True`**: Oculta a senha por padrão na serialização.
- **`validate_default=True`**: Valida até os valores atribuídos por omissão.
- **`model_validator(mode="after")`**: Validações que dependem de múltiplos campos após a criação do objeto.
- **`field_serializer`**: Converte o Enum para string apenas quando exportamos para JSON.
- **`model_serializer`**: Personaliza a estrutura completa do dicionário resultante.

In [4]:
class User(BaseModel):
    name: str = Field(examples=["Example"])
    email: EmailStr = Field(
        examples=["user@arjancodes.com"],
        description="The email address of the user",
        frozen=True,
    )
    password: SecretStr = Field(
        examples=["Password123"], description="The password of the user", exclude=True
    )
    role: Role = Field(
        description="The role of the user",
        examples=[1, 2, 4, 8],
        default=0,
        validate_default=True,
    )

    @field_validator("name")
    @classmethod
    def validate_name(cls, v: str) -> str:
        if not VALID_NAME_REGEX.match(v):
            raise ValueError("Name is invalid")
        return v

    @field_validator("role", mode="before")
    @classmethod
    def validate_role(cls, v: int | str | Role) -> Role:
        op = {int: lambda x: Role(x), str: lambda x: Role[x], Role: lambda x: x}
        try:
            return op[type(v)](v)
        except (KeyError, ValueError):
            raise ValueError(f"Role is invalid")

    @model_validator(mode="before")
    @classmethod
    def validate_user_pre(cls, v: dict[str, Any]) -> dict[str, Any]:
        if "name" not in v or "password" not in v:
            raise ValueError("Name and password are required")
        if v["name"].casefold() in v["password"].casefold():
            raise ValueError("Password cannot contain name")
        if not VALID_PASSWORD_REGEX.match(v["password"]):
            raise ValueError("Password is invalid")
        v["password"] = hashlib.sha256(v["password"].encode()).hexdigest()
        return v

    @model_validator(mode="after")
    def validate_user_post(self, v: Any) -> Self:
        if self.role == Role.Admin and self.name != "Arjan":
            raise ValueError("Only Arjan can be an admin")
        return self

    @field_serializer("role", when_used="json")
    @classmethod
    def serialize_role(cls, v: Role) -> str:
        return v.name

    @model_serializer(mode="wrap", when_used="json")
    def serialize_user(self, serializer, info) -> dict[str, Any]:
        if not info.include and not info.exclude:
            return {"name": self.name, "role": self.role.name}
        return serializer(self)

## 5. Demonstração de Serialização

Aqui testamos como o Pydantic exporta os dados em diferentes formatos (Dicionário vs JSON) e como os nossos serializadores personalizados alteram o resultado.

In [5]:
data = {
    "name": "Arjan",
    "email": "example@arjancodes.com",
    "password": "Password123",
    "role": "Admin",
}

user = User.model_validate(data)
if user:
    print("The serializer that returns a dict:", user.model_dump(), sep="\n", end="\n\n")
    print("The serializer that returns a JSON string:", user.model_dump(mode="json"), sep="\n", end="\n\n")
    print("The serializer that returns a json string, excluding the role:", user.model_dump(exclude=["role"], mode="json"), sep="\n", end="\n\n")
    print("The serializer that encodes all values to a dict:", dict(user), sep="\n")

The serializer that returns a dict:
{'name': 'Arjan', 'email': 'example@arjancodes.com', 'role': <Role.Admin: 4>}

The serializer that returns a JSON string:
{'name': 'Arjan', 'role': 'Admin'}

The serializer that returns a json string, excluding the role:
{'name': 'Arjan', 'email': 'example@arjancodes.com'}

The serializer that encodes all values to a dict:
{'name': 'Arjan', 'email': 'example@arjancodes.com', 'password': SecretStr('**********'), 'role': <Role.Admin: 4>}
